In [ ]:
# ==========================================================
# セル1：ライブラリと共通モジュール fft_common の読み込み
# ==========================================================
# 【このノートブックの目的】
#   全加工対象ファイルに適用するローパスフィルタの
#   カットオフ周波数を決めるための材料を作る。
#
#   指定フォルダ配下の Futaba形式 と NR-500形式 を「まとめて1つの集計」にする。
#   決めたいカットオフは1つなので、2形式を分けずに横断して見るのがポイント。
#
# ★加工対象CSVには一切書き込まない（読み取り専用）★
# ==========================================================
import os
import sys
import time
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog

# --- 共通モジュール fft_common.py を探して読み込む ---
# ノートブックの実行時カレントディレクトリが一定しないため、
# 想定される場所を順に探す。見つからなければ理由を明示して止める。
_here = os.getcwd()
_cands = [
    _here,
    os.path.join(_here, "コードフォルダ"),
    os.path.dirname(_here),
    os.path.join(os.path.dirname(_here), "コードフォルダ"),
]
for _c in _cands:
    if os.path.isfile(os.path.join(_c, "fft_common.py")):
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break
else:
    raise FileNotFoundError(
        "fft_common.py が見つかりません。\n探した場所:\n  "
        + "\n  ".join(_cands)
        + "\nコードフォルダと同じ場所（またはその親フォルダ）で実行してください。"
    )

import fft_common
from fft_common import (
    VOLT_TO_MPA, ZERO_ADJUST_ROWS, bin_to_grid, cumulative_power, folder_tag,
    list_csv_by_format, lowpass_fft, make_log_grid, metrics, psd, read_any,
    setup_japanese_font,
)

setup_japanese_font(plt)

# パーセンタイル集計では、そのファイルの帯域外ビンが NaN になるのは正常。
# （Futaba は 500Hz 超が NaN）そのための警告は出さない。
warnings.filterwarnings("ignore", category=RuntimeWarning,
                        message="All-NaN slice encountered")

print("✅ 準備完了")
print(f"   共通モジュール : {fft_common.__file__}")
print(f"   単位換算       : 電圧[V] × {VOLT_TO_MPA} = 圧力[MPa]（NR-500のみ）")
print(f"   ゼロ点合わせ   : 先頭 {ZERO_ADJUST_ROWS} 行の平均を全行から引く（NR-500のみ）")
print("👉 次の「セル2」を実行してください。")

In [ ]:
# ==========================================================
# セル2：メインフォルダの選択
# ==========================================================
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
main_dir = filedialog.askdirectory(title="解析対象のメインフォルダを選択してください")
root.destroy()

if not main_dir:
    print("⚠️ フォルダ選択がキャンセルされました。次のセルには進まず、やり直してください。")
else:
    print(f"✅ 選択されたメインフォルダ:\n{main_dir}")

In [ ]:
# ==========================================================
# セル3：対象ファイルの列挙（Futaba形式・NR-500形式の両方を対象にする）
# ==========================================================
if not main_dir:
    raise ValueError("メインフォルダが選択されていません。セル2を再実行してください。")

print("🔍 CSVを走査中...（1行目を見て形式を判定します）")
buckets = list_csv_by_format(main_dir)
target_files = buckets["futaba"] + buckets["nr500"]
n_all = sum(len(v) for v in buckets.values())

print(f"\n見つかったCSVファイル: 合計 {n_all} 件")
print(f"   ├ Futaba形式  : {len(buckets['futaba']):5d} 件 ← 処理します")
print(f"   ├ NR-500形式  : {len(buckets['nr500']):5d} 件 ← 処理します")
print(f"   └ それ以外    : {len(buckets['other']):5d} 件 ← 対象外として除外します")

for label, key in (("Futaba形式", "futaba"), ("NR-500形式", "nr500")):
    if buckets[key]:
        print(f"\n【{label} の例】")
        for f in buckets[key][:2]:
            print(" -", f)
        if len(buckets[key]) > 2:
            print(f"   ... (他 {len(buckets[key]) - 2} 件)")

# 除外したファイルも必ず見せる（無言でスキップしない）
if buckets["other"]:
    print("\n【対象外として除外したファイル】")
    for f in buckets["other"][:5]:
        print(" -", os.path.basename(f))
    if len(buckets["other"]) > 5:
        print(f"   ... (他 {len(buckets['other']) - 5} 件)")

if not target_files:
    print("\n⚠️ 解析できるCSVが1件も見つかりませんでした。フォルダを確認してください。")
else:
    print(f"\n✅ 合計 {len(target_files)} 件を1つの集計にまとめます。")
    print("👉 次の「セル4」を実行してください。")

In [ ]:
# ==========================================================
# セル4：解析設定とグループ化
# ==========================================================
if not target_files:
    raise ValueError("処理対象ファイルがありません。セル3を確認してください。")

# ==========================================
# ⚙️ 設定
# ==========================================
# --- 解析方法 ---
USE_EVENT_WINDOW = True    # True: 圧力が立ち上がっているイベント区間だけを解析（推奨）
                           # False: 記録全体を解析（無信号区間のノイズが混入する）
DETREND = True             # FFT前に解析区間の平均を引く（直流成分を消す）
WINDOW = "hann"            # "hann"（漏れが少ない） / "none"（矩形窓）

# --- カットオフ候補（セル8で歪みとノイズ低減を評価する）---
CUTOFF_CANDIDATES = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100]

# --- パーセンタイル帯（セル6・7）---
PERCENTILES = (50, 95)     # 中央値と95パーセンタイル

# --- 共通周波数グリッド（2形式を同じ土俵に載せるため）---
GRID_MIN_HZ = 0.1
GRID_MAX_HZ = 1.0e5        # NR-500 の Nyquist まで。Futaba は 500Hz より上が NaN になる
GRID_PER_DECADE = 60

# --- セル8の掃引に使うファイル数の上限（全部やると重いので抽出する）---
SWEEP_MAX_FILES = 30

OUT_DIR_NAME = "fft_cutoff_analysis"
# ==========================================

output_dir_path = os.path.join(os.getcwd(), OUT_DIR_NAME)
os.makedirs(output_dir_path, exist_ok=True)

# 末端フォルダ（＝温度・板厚などの条件）ごとにグループ化する
group_to_files = defaultdict(list)
for p in target_files:
    group_to_files[folder_tag(os.path.dirname(p), main_dir)].append(p)
group_names = sorted(group_to_files)

GRID_EDGES, GRID_F = make_log_grid(GRID_MIN_HZ, GRID_MAX_HZ, GRID_PER_DECADE)

print("✅ 設定完了")
print(f"📁 出力先        : {output_dir_path}")
print(f"🎯 解析区間      : {'イベント区間のみ' if USE_EVENT_WINDOW else '記録全体'}")
print(f"🎯 窓関数        : {WINDOW} / 直流除去: {DETREND}")
print(f"📊 共通グリッド  : {GRID_F[0]:.2f}〜{GRID_F[-1]:,.0f} Hz "
      f"({len(GRID_F)} ビン, {GRID_PER_DECADE}点/decade)")
print(f"🔧 カットオフ候補: {CUTOFF_CANDIDATES} Hz")
print(f"📦 条件グループ  : {len(group_names)} 件")
for g in group_names[:8]:
    print(f"     - {g}: {len(group_to_files[g])} ファイル")
if len(group_names) > 8:
    print(f"     ... (他 {len(group_names) - 8} グループ)")
print(f"\n⏱️ セル5の目安  : 約 {len(target_files) * 0.6 / 60:.1f} 分"
      f"（{len(target_files)} ファイル × 約0.6秒）")
print("👉 次の「セル5」を実行してください。")

In [ ]:
# ==========================================================
# セル5：全ファイルの指標とスペクトルを一括計算（重い処理はここだけ）
# ==========================================================
# ファイルは1回しか読みません。計算結果をメモリに保持するので、
# セル6〜9のグラフ・表は何度でも即座に描き直せます。
#
# 保持するのは共通グリッド上の値（既定360ビン）だけなので、
# 1000ファイル×2chでも数MBに収まります。
# ==========================================================
print("🚀 一括計算を開始します...")
t0 = time.time()

records = []        # 1ファイル×1チャンネル につき1行の指標
grid_psd = {}       # (パス, ch) -> 共通グリッド上のPSD [MPa^2/Hz]
grid_cum = {}       # (パス, ch) -> 共通グリッド上の累積パワー比率
read_errors = []    # 読み込み自体に失敗したファイル

for i, path in enumerate(target_files, 1):
    fname = os.path.basename(path)
    group = folder_tag(os.path.dirname(path), main_dir)
    try:
        df, dt, info = read_any(path)
    except Exception as e:
        read_errors.append((path, f"{type(e).__name__}: {e}"))
        continue

    for ch in df.columns:
        y = df[ch].to_numpy(dtype=np.float64)
        try:
            m, (f, P), (i0, i1) = metrics(
                y, dt,
                use_event=USE_EVENT_WINDOW, detrend=DETREND, window=WINDOW,
                zero_rows=ZERO_ADJUST_ROWS if info["zero_applied"] else None,
            )
        except Exception as e:
            read_errors.append((path, f"[{ch}] {type(e).__name__}: {e}"))
            continue

        key = (path, ch)
        grid_psd[key] = bin_to_grid(f, P, GRID_EDGES)

        # 累積パワー比率も共通グリッドへ載せる（帯域外は NaN のまま）
        c = cumulative_power(P)
        cg = np.full(len(GRID_F), np.nan)
        if len(f) > 2:
            inb = (GRID_F >= f[1]) & (GRID_F <= f[-1])
            cg[inb] = np.interp(GRID_F[inb], f[1:], c[1:])
        grid_cum[key] = cg

        rec = {"file": fname, "path": path, "group": group,
               "format": info["format"], "channel": ch,
               "zero_applied": info["zero_applied"],
               "zero_offset_applied_MPa": info["zero_offset"].get(ch, 0.0)}
        rec.update(m)
        records.append(rec)

    if i % 25 == 0 or i == len(target_files):
        el = time.time() - t0
        rest = el / i * (len(target_files) - i)
        print(f"  {i}/{len(target_files)} ファイル  "
              f"（経過 {el:.0f}秒 / 残り約 {rest:.0f}秒）")

met = pd.DataFrame(records)

print("\n" + "-" * 56)
print(f"🏁 一括計算 完了！  {len(met)} 行（ファイル×チャンネル）  "
      f"所要 {time.time() - t0:.0f} 秒")

# ---- 品質フラグの集計（無言でスキップしない）----
print("\n【品質フラグの内訳】")
if len(met):
    flag_counts = met["flags"].value_counts()
    for flag, cnt in flag_counts.items():
        mark = "✅" if flag == "ok" else "⚠️"
        print(f"  {mark} {flag:34s} {cnt:5d} 件")
    bad = met[met["flags"] != "ok"]
    if len(bad):
        print(f"\n  ⚠️ 要確認 {len(bad)} 件（先頭10件）:")
        for _, r in bad.head(10).iterrows():
            print(f"     {r['file']} [{r['channel']}] → {r['flags']}")
        if len(bad) > 10:
            print(f"     ... (他 {len(bad) - 10} 件。全件は セル9 のCSVで確認できます)")

if read_errors:
    print(f"\n❌ 読み込み失敗 {len(read_errors)} 件:")
    for p, msg in read_errors[:10]:
        print(f"   {os.path.basename(p)}: {msg}")

pts = sum(len(v) for v in grid_psd.values()) * 2
print(f"\n💾 キャッシュ: {len(grid_psd)} 系列 × {len(GRID_F)} ビン "
      f"（約 {pts * 8 / 1e6:.1f} MB）")
print("👉 次の「セル6」を実行してください。")

In [ ]:
# ==========================================================
# セル6：PSDのパーセンタイル帯グラフ（両対数）
# ==========================================================
# 【なぜ両対数か】
#   判断すべき 5〜50Hz 帯の成分は 0〜5MPa のリニア軸では全部ゼロに見える。
#   カットオフは「信号がノイズフロアに沈む点」を見て決めるので対数軸が必須。
#
# 【なぜ振幅でなくPSDか】
#   Futaba(Δf=0.05Hz) と NR-500(Δf=0.2Hz) では振幅スペクトルの高さが
#   分解能に依存してしまい直接比較できない。PSD[MPa^2/Hz]なら比較できる。
# ==========================================================
print("🚀 PSDのパーセンタイル帯グラフを作成します...")
p_lo, p_hi = PERCENTILES


def _stack(keys):
    """指定系列を共通グリッド上に積み上げる。"""
    if not keys:
        return None
    return np.vstack([grid_psd[k] for k in keys])


def _draw_band(ax, arr, color, label):
    """中央値と上側パーセンタイルの帯を描く。"""
    with np.errstate(all="ignore"):
        med = np.nanpercentile(arr, p_lo, axis=0)
        hi = np.nanpercentile(arr, p_hi, axis=0)
    ax.fill_between(GRID_F, med, hi, color=color, alpha=0.20, linewidth=0)
    ax.plot(GRID_F, med, color=color, linewidth=1.6,
            label=f"{label} 中央値 (n={arr.shape[0]})")
    ax.plot(GRID_F, hi, color=color, linewidth=0.9, linestyle="--",
            label=f"{label} {p_hi}パーセンタイル")
    return med


def _finish(ax, title, subtitle=""):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel("PSD [MPa$^2$/Hz]")
    ax.set_title(title)
    if subtitle:
        # タイトルと重ならないよう軸の下側に置く
        ax.text(0.5, -0.155, subtitle, transform=ax.transAxes, ha="center",
                fontsize=9, color="dimgray")
    ax.grid(True, which="both", linestyle="--", alpha=0.4)
    ax.legend(loc="upper right", fontsize=8)


keys_all = list(grid_psd)
if not keys_all:
    raise ValueError("スペクトルがありません。セル5を実行してください。")

# ---- (1) 全体：形式ごとに分けて描く ----
# Futaba は 50Hz より上で PSD が平坦になる（エイリアシングかノイズフロア）。
# 形式を分けて描くと、その違いが一目で分かる。
fig, ax = plt.subplots(figsize=(11, 6))
for fmt, color in (("futaba", "tab:blue"), ("nr500", "tab:orange")):
    keys = [(r["path"], r["channel"]) for _, r in met.iterrows() if r["format"] == fmt]
    arr = _stack(keys)
    if arr is not None:
        _draw_band(ax, arr, color, fmt)

for q, col, ls in (("f99_Hz", "tab:green", "-."), ("f999_Hz", "tab:red", ":")):
    v = float(met[q].median())
    ax.axvline(v, color=col, linestyle=ls, linewidth=1.3,
               label=f"{q.replace('_Hz','')} 中央値 = {v:.1f} Hz")
ax.legend(loc="upper right", fontsize=8)
_finish(ax, "PSD パーセンタイル帯 － 全ファイル（形式別）",
        "Futaba は Nyquist 500Hz。NR-500 のみ 500Hz 以上に線が伸びる")
fig.tight_layout()
fig.savefig(os.path.join(output_dir_path, "psd_percentile_ALL.png"), dpi=150)
plt.show()

# ---- (2) 条件グループごと ----
n_saved = 1
for g in group_names:
    keys = [(r["path"], r["channel"]) for _, r in met.iterrows() if r["group"] == g]
    arr = _stack(keys)
    if arr is None:
        continue
    fig, ax = plt.subplots(figsize=(11, 5.5))
    _draw_band(ax, arr, "tab:blue", g)
    sub = met[met["group"] == g]
    for q, col, ls in (("f99_Hz", "tab:green", "-."), ("f999_Hz", "tab:red", ":")):
        v = float(sub[q].median())
        ax.axvline(v, color=col, linestyle=ls, linewidth=1.3,
                   label=f"{q.replace('_Hz','')} 中央値 = {v:.1f} Hz")
    ax.legend(loc="upper right", fontsize=8)
    _finish(ax, f"PSD パーセンタイル帯 － {g}")
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir_path, f"psd_percentile_{g}.png"), dpi=150)
    plt.close(fig)
    n_saved += 1

print(f"🏁 完了！ {n_saved} 枚を保存しました → {output_dir_path}")
print("👉 次の「セル7」を実行してください。")

In [ ]:
# ==========================================================
# セル7：累積パワー曲線（カットオフを直接読み取るためのグラフ）
# ==========================================================
# 「周波数 f 以下に信号パワーの何%が入っているか」を描く。
#   99% 線・99.9% 線との交点が、その条件で信号を保つのに必要な帯域の目安。
#   カットオフはこの交点より上に置く必要がある。
# ==========================================================
print("🚀 累積パワー曲線を作成します...")

keys_all = list(grid_cum)
arr_all = np.vstack([grid_cum[k] for k in keys_all])
p_lo, p_hi = PERCENTILES


def _cum_plot(ax, arr, color, label):
    with np.errstate(all="ignore"):
        med = np.nanpercentile(arr, 50, axis=0)
        lo = np.nanpercentile(arr, 100 - p_hi, axis=0)
        hi = np.nanpercentile(arr, p_hi, axis=0)
    ax.fill_between(GRID_F, lo, hi, color=color, alpha=0.20, linewidth=0)
    ax.plot(GRID_F, med, color=color, linewidth=1.8,
            label=f"{label} 中央値 (n={arr.shape[0]})")


def _cum_finish(ax, title):
    for q, col in ((0.99, "tab:green"), (0.999, "tab:red")):
        ax.axhline(q, color=col, linestyle="--", linewidth=1.1)
        ax.text(GRID_F[0] * 1.2, q, f"  {q * 100:g}%", color=col,
                va="bottom", fontsize=9)
    ax.set_xscale("log")
    ax.set_xlim(GRID_F[0], 1000)
    ax.set_ylim(0.80, 1.002)
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel("累積パワー比率")
    ax.set_title(title)
    ax.grid(True, which="both", linestyle="--", alpha=0.4)
    ax.legend(loc="lower right", fontsize=8)


# ---- (1) 全体：形式別 ----
fig, ax = plt.subplots(figsize=(11, 6))
for fmt, color in (("futaba", "tab:blue"), ("nr500", "tab:orange")):
    keys = [(r["path"], r["channel"]) for _, r in met.iterrows() if r["format"] == fmt]
    if keys:
        _cum_plot(ax, np.vstack([grid_cum[k] for k in keys]), color, fmt)
_cum_finish(ax, "累積パワー曲線 － 全ファイル（形式別）")
fig.tight_layout()
fig.savefig(os.path.join(output_dir_path, "cumulative_power_ALL.png"), dpi=150)
plt.show()

# ---- (2) 条件グループごと ----
n_saved = 1
for g in group_names:
    keys = [(r["path"], r["channel"]) for _, r in met.iterrows() if r["group"] == g]
    if not keys:
        continue
    fig, ax = plt.subplots(figsize=(11, 5.5))
    _cum_plot(ax, np.vstack([grid_cum[k] for k in keys]), "tab:blue", g)
    _cum_finish(ax, f"累積パワー曲線 － {g}")
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir_path, f"cumulative_power_{g}.png"), dpi=150)
    plt.close(fig)
    n_saved += 1

# ---- 数値サマリ ----
print(f"\n🏁 完了！ {n_saved} 枚を保存しました")
print("\n【信号を保つのに必要な帯域（全ファイル集計）】")
print(f"{'指標':>14} | {'中央値':>9} | {'95%tile':>9} | {'最大':>9}")
print("-" * 52)
for col, name in (("f99_Hz", "f99"), ("f999_Hz", "f99.9"),
                  ("required_bw_Hz", "立上りからの必要帯域")):
    s = met[col].dropna()
    if len(s):
        print(f"{name:>14} | {s.median():7.2f}Hz | {s.quantile(0.95):7.2f}Hz "
              f"| {s.max():7.2f}Hz")
print("\n👉 次の「セル8」を実行してください。")

In [ ]:
# ==========================================================
# セル8：カットオフ候補の掃引（歪み と ノイズ低減 のトレードオフ）
# ==========================================================
# 候補ごとに実際にローパスをかけてみて、
#   ・ピーク圧力がどれだけ削られるか（歪み。小さいほど良い）
#   ・ベースラインノイズがどれだけ減るか（大きいほど良い）
# を測る。ノイズ低減が頭打ちになる手前が最適なカットオフ。
#
# ※ここでかけるのは評価用のゼロ位相FFTローパス。
#   ファイルへの実適用は別途（scipy.signal.butter + filtfilt を推奨）。
# ※元ファイルには何も書き込まない。
# ==========================================================
from fft_common import find_event, robust_baseline, robust_sigma

print("🚀 カットオフ候補の掃引を開始します...")

# 条件グループから均等に抽出する（全ファイルでやると重いため）
_rng = np.random.default_rng(0)
per_group = max(1, SWEEP_MAX_FILES // max(len(group_names), 1))
sweep_files = []
for g in group_names:
    fs = sorted(group_to_files[g])
    if len(fs) <= per_group:
        sweep_files += fs
    else:
        sweep_files += [fs[i] for i in
                        sorted(_rng.choice(len(fs), per_group, replace=False))]
sweep_files = sweep_files[:SWEEP_MAX_FILES]

print(f"   抽出: {len(sweep_files)} ファイル（{len(group_names)} グループから均等に）")
print(f"   候補: {CUTOFF_CANDIDATES} Hz")
print(f"⏱️ 目安 約 {len(sweep_files) * 0.6 * (1 + len(CUTOFF_CANDIDATES) * 0.3) / 60:.1f} 分\n")

rows = []
t0 = time.time()
for i, path in enumerate(sweep_files, 1):
    try:
        df, dt, info = read_any(path)
    except Exception as e:
        print(f"   [スキップ] {os.path.basename(path)}: {e}")
        continue

    for ch in df.columns:
        y = df[ch].to_numpy(dtype=np.float64)
        base, _ = robust_baseline(y)
        i0, i1, _flags, _info = find_event(y, dt)
        yc = y - base                      # 端の不連続を減らすため基準線を引いてから濾波
        pk0 = float(yc.max())
        if pk0 <= 0:
            continue

        # ノイズはイベント「前」の静穏部で測る（イベント後は残圧が残るため）
        nz = yc[:i0] if i0 > 200 else yc[i1 + 1:]
        if len(nz) < 200:
            continue
        rms0 = robust_sigma(nz)      # MADが0になる量子化データにも対応

        for fc in CUTOFF_CANDIDATES:
            z = lowpass_fft(yc, dt, fc)
            zn = z[:i0] if i0 > 200 else z[i1 + 1:]
            rms = robust_sigma(zn)
            rows.append({
                "file": os.path.basename(path), "group":
                    folder_tag(os.path.dirname(path), main_dir),
                "format": info["format"], "channel": ch, "cutoff_Hz": fc,
                "peak_error_pct": 100.0 * (float(z.max()) - pk0) / pk0,
                "noise_reduction_x": (rms0 / rms) if rms > 0 else np.nan,
            })

    if i % 5 == 0 or i == len(sweep_files):
        el = time.time() - t0
        print(f"  {i}/{len(sweep_files)} ファイル（経過 {el:.0f}秒）")

sweep = pd.DataFrame(rows)
if sweep.empty:
    raise ValueError("掃引結果が空です。対象ファイルを確認してください。")

# ---- 集計表 ----
agg = sweep.groupby("cutoff_Hz").agg(
    ピーク誤差_中央値=("peak_error_pct", "median"),
    ピーク誤差_最悪=("peak_error_pct", "min"),      # 負に大きいほど悪い
    ノイズ低減_中央値=("noise_reduction_x", "median"),
    ノイズ低減_最小=("noise_reduction_x", "min"),
).reset_index()

print("\n" + "=" * 74)
print("【カットオフ掃引の結果】")
print(f"{'カットオフ':>10} | {'ピーク誤差(中央)':>15} | {'ピーク誤差(最悪)':>15} | "
      f"{'ノイズ低減(中央)':>15}")
print("-" * 74)
for _, r in agg.iterrows():
    print(f"{r['cutoff_Hz']:8.0f}Hz | {r['ピーク誤差_中央値']:+14.2f}% | "
          f"{r['ピーク誤差_最悪']:+14.2f}% | {r['ノイズ低減_中央値']:14.1f}倍")
print("=" * 74)

# ノイズ低減が頭打ちになる点を探す（最大値の95%に初めて達する候補）
best_nr = agg["ノイズ低減_中央値"].max()
knee = agg[agg["ノイズ低減_中央値"] >= 0.95 * best_nr]["cutoff_Hz"].min()
print(f"\n📈 ノイズ低減が頭打ちになるのは {knee:.0f} Hz 付近"
      f"（最大 {best_nr:.1f}倍 の95%に到達）")
print("👉 次の「セル9」を実行してください。")

In [ ]:
# ==========================================================
# セル9：CSV出力とカットオフ周波数の要約
# ==========================================================
MAX_PEAK_ERROR_PCT = 2.0     # 目標とするピーク誤差（絶対値・%）

csv1 = os.path.join(output_dir_path, "metrics_per_file.csv")
csv2 = os.path.join(output_dir_path, "cutoff_sweep.csv")
csv3 = os.path.join(output_dir_path, "cutoff_summary.csv")

met.to_csv(csv1, index=False, encoding="utf-8-sig")
sweep.to_csv(csv2, index=False, encoding="utf-8-sig")
agg.to_csv(csv3, index=False, encoding="utf-8-sig")

print("💾 CSVを保存しました（Excelでそのまま開けます）")
print(f"   metrics_per_file.csv  … {len(met):6d} 行（1ファイル×1チャンネルの全指標）")
print(f"   cutoff_sweep.csv      … {len(sweep):6d} 行（掃引の生データ）")
print(f"   cutoff_summary.csv    … {len(agg):6d} 行（掃引の集計）")

# ---- 下限：これを下回ると信号そのものが削れる ----
f999_p95 = float(met["f999_Hz"].quantile(0.95))
bw_p95 = float(met["required_bw_Hz"].dropna().quantile(0.95))
need = max(f999_p95, bw_p95)

# ---- 推奨：下限を満たす最小の候補 ----
cand = agg[agg["cutoff_Hz"] >= need]
rec = float(cand["cutoff_Hz"].min()) if len(cand) else float("nan")

# ---- 到達できるピーク誤差の下限（カットオフをいくら上げても残る分）----
floor_err = abs(float(agg["ピーク誤差_最悪"].iloc[-1]))

print("\n" + "=" * 72)
print("【カットオフ周波数の要約】")
print("=" * 72)
print(f"  解析ファイル数        : {met['file'].nunique()} 件"
      f"（{len(met)} 系列 / 形式 {met['format'].nunique()} 種）")
print(f"  品質フラグ ok の割合  : {100 * (met['flags'] == 'ok').mean():.1f} %")
print()
print("  ── 下限（信号を壊さないために必要）──")
print(f"  f99.9 の95パーセンタイル  : {f999_p95:7.2f} Hz   信号パワーの99.9%がここ以下")
print(f"  立上りからの必要帯域(95%) : {bw_p95:7.2f} Hz   パルスの形を保つのに必要")
print(f"  → カットオフの下限        : {need:7.2f} Hz")
print()
print("  ── 上限側（これ以上上げても得がない）──")
print(f"  ノイズ低減が頭打ちになる点: {knee:7.2f} Hz")
print()

if np.isfinite(rec):
    r = agg[agg["cutoff_Hz"] == rec].iloc[0]
    print(f"  ★ 推奨カットオフ: {rec:.0f} Hz")
    print(f"       ピーク誤差 : 中央 {r['ピーク誤差_中央値']:+.2f}% / "
          f"最悪 {r['ピーク誤差_最悪']:+.2f}%")
    print(f"       ノイズ低減 : 中央 {r['ノイズ低減_中央値']:.1f}倍 / "
          f"最小 {r['ノイズ低減_最小']:.1f}倍")
    worst = abs(float(r["ピーク誤差_最悪"]))
    if worst <= MAX_PEAK_ERROR_PCT:
        print(f"       → 目標のピーク誤差 {MAX_PEAK_ERROR_PCT}% 以内を満たしています。")
    elif worst - floor_err <= 1.0:
        print(f"       → 最悪値 {worst:.2f}% は目標 {MAX_PEAK_ERROR_PCT}% を超えますが、")
        print(f"          カットオフを最大 {agg['cutoff_Hz'].max():.0f}Hz まで上げても "
              f"{floor_err:.2f}% までしか下がりません。")
        print("          一部のチャンネルのピークが元々鋭く、どんなローパスでも削れるためです。")
        print("          その系列は cutoff_sweep.csv で特定できます（不良ショットの可能性も）。")
    else:
        print(f"       → 最悪値 {worst:.2f}% が目標 {MAX_PEAK_ERROR_PCT}% を超えています。")
        print("          もう少し高い候補を選ぶか、外れ値の系列を確認してください。")
else:
    print(f"  ⚠️ 下限 {need:.1f} Hz 以上の候補がありません。")
    print("     CUTOFF_CANDIDATES にもっと高い値を足してセル8から再実行してください。")
print("=" * 72)

print("\n【この数字をそのまま使う前に確認してください】")
print(" 1. セル6のPSDグラフで、推奨値より上が本当にノイズフロアになっているか")
print(" 2. セル7の累積パワー曲線で、99.9%線との交点が推奨値より左にあるか")
print(" 3. セル5の品質フラグに ok 以外が多くないか（多い場合は該当ファイルを確認）")
print(" 4. Futaba形式は 50Hz より上の PSD が平坦になりやすく")
print("    （エイリアシングまたはノイズフロア）、高周波側の判断には使えません。")
print("    高周波側は NR-500形式の線を見てください。")

print("\n【次のステップ：実際にローパスをかけるとき】")
print(" ・scipy が未インストールです。`uv add scipy` で追加してください。")
print(" ・このノートブックのFFTローパスは評価用です。実適用には")
print("   scipy.signal.butter + filtfilt（ゼロ位相）を使ってください。")
print("   ゼロ位相でないとピーク圧力の発生時刻がずれます。")
print("\n🎉 すべての処理が完了しました。")